# eICU

In [ ]:

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import dvgate as dg
import eicu_loader as L

SEED = 99
PARQUET = "eicu_features_24h.parquet"
N_PLAYERS, MIN_ENC, CAP, N_CORRUPT = 20, 400, 500, 5
PILOT, T_OBJETIVO, N_EXACT, MINUTOS_REF = 45, 230, 10, 30


ESCALERA = {"C0_limpio": (None, 0.0), "C1_bal_0.50": ("balanced", 0.50),
            "C2_bal_1.00": ("balanced", 1.00), "C3_rand_0.50": ("random", 0.50)}
T_INI, tiempos = time.time(), {}
print("umbrales:", dg.THR)

## 1 — Carga y fuentes a corromper

In [ ]:
t = time.time()
D = L.load(PARQUET, N_PLAYERS, MIN_ENC, seed=SEED)
X, y, src, P = D["X"], D["y"], D["src"], D["players"]
TR, VA, TE = D["idx_tr"], D["idx_va"], D["idx_te"]
MASK = np.sort(np.concatenate([TR, VA]))         


_rng = np.random.default_rng(SEED)
CORRUPT = sorted(int(_rng.choice(g))
                 for g in np.array_split(D["meta"].n_filas.sort_values().index.values, N_CORRUPT))
tiempos["carga"] = (time.time() - t) / 60

print(f"fuentes a corromper: "
      f"{D['meta'].n_filas[CORRUPT].to_dict()}   ({tiempos['carga'] * 60:.0f}s)")

## 2 — Coste medido 

In [ ]:
t = time.time()
v_tmp, _, idxp = dg.make_game(X, y, src, TR, VA, P, CAP, SEED)
_, _, _, nev = dg.tmc_shapley(P, v_tmp, 5, SEED)
C_EVAL = (time.time() - t) / nev


t = time.time(); dg.eval_test(X, y, src, TR, TE, P, SEED); C_FIT = time.time() - t
n, nc = len(P), len(ESCALERA)

T_FULL = T_OBJETIVO

PRESU = dg.budget_report(n, nc, PILOT, T_FULL, N_EXACT, C_EVAL, C_FIT)
PREV = PRESU.min_total.sum() + tiempos["carga"]

print(f"coste: evaluación {C_EVAL:.4f}s,  completo {C_FIT:.4f}s | filas del "
      f"juego: {sum(len(i) for i in idxp.values()):,} | T = {T_FULL} (fijado, NO recortado)\n"
      f"{PRESU.round(2).to_string(index=False)}\n"
      f"PROYECCIÓN {PREV:.0f} min, más ~{nc * 0.3:.0f} de bootstrap  "
      f"Referencia : {MINUTOS_REF} min: "
      f"{'DENTRO' if PREV <= MINUTOS_REF else 'POR ENCIMA — se respeta T. Aborta aquí si no quieres gastarlo.'}")

## 3 Panel 

In [ ]:
def run_condition(nombre, mode, frac):
    t0 = time.time()
    yc = y if mode is None else dg.corrupt_labels(y, src, CORRUPT, frac, SEED, MASK, mode)
    meta = dg.source_meta(yc, src, D["frac_na"], TR, P)
    v, v0, _ = dg.make_game(X, yc, src, TR, VA, P, CAP, SEED)

    t = time.time()
    pan = dg.run_panel(X, yc, src, TR, VA, TE, P, v, meta, SEED, PILOT, T_FULL)
    min_panel = (time.time() - t) / 60

    t = time.time()                               
    phi, se, _, _ = dg.tmc_shapley(P, v, T_FULL, SEED)
    lo = dg.loo(P, v)
    min_caro = (time.time() - t) / 60
    cl, guiada = dg.close_loop(X, yc, src, TR, TE, P, phi, se, meta, SEED)

    val = pd.DataFrame({"phi": pd.Series(phi), "se": pd.Series(se), "loo": pd.Series(lo),
                        "lift_out": pan["daño"].lift_out, "lift_self": pan["daño"].lift_self,
                        "ret": pan["daño"].ret, "asimetria": pan["monotonia"].asimetria,
                        "frac_neg": pan["monotonia"].frac_neg,
                        "marcada_panel": pan["daño"]["dañina"],
                        "trivial": pan["trivial"].trivial}).assign(condicion=nombre)
    val["corrompida"] = val.index.isin(CORRUPT)


    def pr(sel):                                   
        s = set(val.index[sel])
        return len(s & set(CORRUPT)) / max(len(s), 1), len(s & set(CORRUPT)) / len(CORRUPT)
    det = {f"{q}_{k}": r for k, sel in [("panel", val.marcada_panel),
                                        ("caro", val.index.isin(guiada)),
                                        ("trivial", val.trivial)]
           for q, r in zip(("prec", "exh"), pr(sel))}

    print(f"\n{'=' * 76}\n{nombre}   v(vacío)={v0:.4f}  v(N)={v(P):.4f}  "
          f"excedente={v(P) - v0:.4f}  reparto medio={(v(P) - v0) / n:.5f}\n"
          f"{pan['tabla'].to_string(index=False)}\n"
          f"PANEL: {pan['veredicto']} — {pan['motivo']}  ({min_panel * 60:.0f}s)\n"
          f"CARO ({min_caro:.1f} min): phi<0 en {sorted(val.index[val.phi < 0])} | "
          f"regla phi+2SE<0 marca {guiada} | LOO<0 en {sorted(val.index[val.loo < 0])}\n"
          f"detección prec/exh — panel {det['prec_panel']:.2f}/{det['exh_panel']:.2f} | "
          f"caro {det['prec_caro']:.2f}/{det['exh_caro']:.2f} | "
          f"trivial {det['prec_trivial']:.2f}/{det['exh_trivial']:.2f}\n"
          f"{cl[['k', 'contexto', 'auroc', 'ap', 'd_ap', 'd_ap_lo', 'd_ap_hi']].round(4).to_string()}")

    caro = "PROCEDE" if guiada else "NO PROCEDE"
    return (dict(condicion=nombre, excedente=v(P) - v0, panel=pan["veredicto"],
                 esperado="NO PROCEDE" if mode is None else "PROCEDE",
                 caro=caro, concuerda=pan["veredicto"] == caro, motivo=pan["motivo"],
                 mde_ap=pan["potencia"]["mde_ap"],
                 rho_prometido=pan["resolucion"]["rho_esperado"],
                 perm_necesarias=pan["resolucion"]["perm_necesarias"],
                 min_panel=min_panel, min_caro=min_caro, **det),
            val, cl.assign(condicion=nombre), phi, se)

## 4 — Corrupcion de etiquetas

In [ ]:
ver, vals, cierres, phis, ses = [], [], [], {}, {}
for nombre, (mode, frac) in ESCALERA.items():
    r, val, cl, phi, se = run_condition(nombre, mode, frac)
    ver.append(r); vals.append(val); cierres.append(cl)
    phis[nombre], ses[nombre] = phi, se
VER, VALS, CIERRES = pd.DataFrame(ver).set_index("condicion"), pd.concat(vals), pd.concat(cierres)

## 5 Shapley exacto sobre 10 fuenets

In [ ]:
t = time.time()
SUB = sorted(np.random.default_rng(SEED + 1).choice(P, N_EXACT, replace=False).tolist())

v_sub, _, _ = dg.make_game(X, y, src, TR, VA, SUB, CAP, SEED)
ex = dg.exact_shapley(SUB, v_sub)
tm, se_s, _, _ = dg.tmc_shapley(SUB, v_sub, T_FULL, SEED)

EXACTO = pd.DataFrame({"exacto": pd.Series(ex), "tmc": pd.Series(tm), "se": pd.Series(se_s)})
EXACTO["error"] = (EXACTO.exacto - EXACTO.tmc).abs()

rho = float(np.corrcoef(EXACTO.exacto.rank(), EXACTO.tmc.rank())[0, 1])
tiempos["exacto"] = (time.time() - t) / 60

print(f"{EXACTO.round(5).to_string()}\n"
      f"rho(exacto, TMC)={rho:.3f} | error máx={EXACTO.error.max():.5f} | "
      f"SE medio={EXACTO.se.mean():.5f} | eficiencia: suma - (v(N)-v(vacío))="
      f"{sum(ex.values()) - (v_sub(SUB) - v_sub([])):+.1e}  ({tiempos['exacto']:.1f} min)\n"
      f" rho={VER.loc['C0_limpio', 'rho_prometido']:.3f}, "
      f"el exacto da {rho:.3f}")

## 6 — Estabilidad de las fuentes limpias , stability_check

V_fixed es lo que nos da el problema, por eos lo miramos

In [ ]:
EST = dg.stability_check(phis, ses, CORRUPT, "C0_limpio", VER.excedente.to_dict())
print(f"{EST.round(4).to_string()}\nlimpias sin negatividad significativa: "
      f"{'SÍ' if bool((EST.n_neg_significativa == 0).all()) else 'NO — el contraste entre condiciones NO es interpretable'}")

fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
for i, (nom, phi) in enumerate(phis.items()):
    for g, c, m, et in [(0, "tab:blue", "o", "limpias"), (1, "tab:red", "x", "corrompidas")]:
        w = [phi[p] for p in P if (p in CORRUPT) == bool(g)]
        ax[0].scatter([i] * len(w), w, c=c, marker=m, s=26, label=et if not i else "")

ax[1].errorbar(range(n), [phis["C0_limpio"][p] for p in P],
               yerr=[ses["C0_limpio"][p] for p in P], fmt="o", ms=3, lw=.8)

ax[0].set_xticks(range(len(phis))); ax[0].set_xticklabels(phis.keys(), rotation=15)
ax[0].set_title("Escalera: valores por condición"); ax[0].legend()
ax[0].set_ylabel(r"$\hat\varphi_h$"); ax[1].set_xlabel("hospital")
ax[1].set_title("C0 limpio: valores con 1 SE de Monte Carlo")

for a in ax: a.axhline(0, c="k", lw=.8)
fig.tight_layout(); fig.savefig("fig_escalera.png", dpi=130)

## 7 exportamos...

In [ ]:
tiempos["total"] = (time.time() - T_INI) / 60
COLS = ["esperado", "panel", "caro", "concuerda"] + \
       [f"{q}_{k}" for k in ("panel", "caro", "trivial") for q in ("prec", "exh")] + \
       ["min_panel", "min_caro"]
print("TABLA FINAL DE VEREDICTOS\n" + VER[COLS].round(3).to_string()
      + "\n\nmotivos:\n" + "\n".join(f"  {c}: {m}" for c, m in VER.motivo.items()))
print(f"\nconcordancia: {int(VER.concuerda.sum())}/{len(VER)} condiciones\n"
      f"panel {VER.min_panel.sum():.2f} min frente a {VER.min_caro.sum():.2f} del cálculo "
      f"caro: factor {VER.min_caro.sum() / VER.min_panel.sum():.1f}x a n={n}\n"
      f"extrapolación a n=60 (TMC medido en el TFM anterior, 32,09 min): factor "
      f"~{32.09 / (VER.min_panel.mean() * 3):.0f}x\n"
      f"presupuesto PREVISTO {PRESU.min_total.sum() + tiempos['carga']:.1f} min | "
      f"REAL {tiempos['total']:.1f} min | referencia {MINUTOS_REF} min\n"
      f"{pd.Series(tiempos).round(2).to_string()}")

for nom, obj in [("veredictos", VER), ("valores", VALS), ("cierre_bucle", CIERRES),
                 ("exacto_vs_tmc", EXACTO), ("estabilidad", EST)]:
    obj.to_csv(f"{nom}.csv")
pd.concat([PRESU.assign(tipo="previsto"),
           pd.DataFrame([dict(bloque=k, min_total=v, tipo="real") for k, v in tiempos.items()])
           ]).to_csv("presupuesto.csv", index=False)
print("ficheros csv: veredictos.csv valores.csv cierre_bucle.csv exacto_vs_tmc.csv "
      "estabilidad.csv presupuesto.csv fig_escalera.png")